In [2]:
import torch
import torch.nn as nn

text = "hello world"

In [3]:
itos = {}
stoi = {}
unsorted_char = set()
for i in text:
    unsorted_char.add(i)
n = len(unsorted_char)
chars = sorted(unsorted_char)
for i , char in enumerate(chars):
    itos[i] = char
    stoi[char] = i

starts = [0, 6]
sequence_length = 4

encoded = []
for char in text:
    encoded.append(stoi[char])

data = torch.tensor(encoded, dtype=torch.long)
x = []
for i in starts:
    tmp = torch.tensor(encoded[i:i+4], dtype= torch.long)
    x.append(tmp)
y = []
for i in starts:
    tmp = torch.tensor(encoded[i+1:i+5], dtype= torch.long)
    y.append(tmp)

x_batch = torch.stack(x, dim=0)
y_batch = torch.stack(y, dim=0)

print(x_batch)
print(y_batch)
print(x_batch.shape, y_batch.shape)


tensor([[3, 2, 4, 4],
        [7, 5, 6, 4]])
tensor([[2, 4, 4, 5],
        [5, 6, 4, 1]])
torch.Size([2, 4]) torch.Size([2, 4])


In [4]:
table = nn.Embedding(n,n)
pred = table(x_batch)
print(pred)
print(x_batch.shape, pred.shape, table.weight.shape)

print(torch.allclose(pred[0,2] , table.weight[x_batch[0,2]]))

tensor([[[-1.6709e+00,  8.9456e-01,  5.3523e-01,  2.1828e-03, -9.7856e-01,
           5.5735e-01, -9.9596e-02,  6.4505e-01],
         [ 2.6014e+00,  2.0782e-01, -6.5115e-02, -1.6696e+00,  5.5539e-01,
           6.2909e-01, -1.1027e+00,  1.5124e-01],
         [ 2.5172e-01, -4.5287e-01,  1.9549e+00, -6.9494e-01,  3.3316e-01,
          -8.5510e-01, -3.0372e-01,  1.2265e+00],
         [ 2.5172e-01, -4.5287e-01,  1.9549e+00, -6.9494e-01,  3.3316e-01,
          -8.5510e-01, -3.0372e-01,  1.2265e+00]],

        [[ 9.5662e-01, -6.8248e-01, -6.4263e-01, -3.9258e-01,  1.6404e+00,
          -5.8185e-01,  8.4497e-01, -1.3400e+00],
         [-3.7676e-01, -8.2829e-01,  1.1617e+00,  2.6402e+00,  5.8183e-01,
           2.7263e-01, -6.5718e-01, -1.4219e+00],
         [ 1.4454e-01, -9.1123e-01,  1.9833e-01,  1.1301e+00, -3.2812e-01,
           7.5871e-01, -7.4364e-02,  5.5488e-01],
         [ 2.5172e-01, -4.5287e-01,  1.9549e+00, -6.9494e-01,  3.3316e-01,
          -8.5510e-01, -3.0372e-01,  1.2265e+00]

In [5]:
flat_logits = pred.reshape(-1,8)
print(flat_logits)
flat_target = y_batch.reshape(-1)
print(flat_target)

loss_api = nn.CrossEntropyLoss()
loss = loss_api(flat_logits,flat_target)
print(torch.allclose(flat_logits[4], pred[1, 0]))
print(flat_target[4] == y_batch[1, 0])
print(loss)

tensor([[-1.6709e+00,  8.9456e-01,  5.3523e-01,  2.1828e-03, -9.7856e-01,
          5.5735e-01, -9.9596e-02,  6.4505e-01],
        [ 2.6014e+00,  2.0782e-01, -6.5115e-02, -1.6696e+00,  5.5539e-01,
          6.2909e-01, -1.1027e+00,  1.5124e-01],
        [ 2.5172e-01, -4.5287e-01,  1.9549e+00, -6.9494e-01,  3.3316e-01,
         -8.5510e-01, -3.0372e-01,  1.2265e+00],
        [ 2.5172e-01, -4.5287e-01,  1.9549e+00, -6.9494e-01,  3.3316e-01,
         -8.5510e-01, -3.0372e-01,  1.2265e+00],
        [ 9.5662e-01, -6.8248e-01, -6.4263e-01, -3.9258e-01,  1.6404e+00,
         -5.8185e-01,  8.4497e-01, -1.3400e+00],
        [-3.7676e-01, -8.2829e-01,  1.1617e+00,  2.6402e+00,  5.8183e-01,
          2.7263e-01, -6.5718e-01, -1.4219e+00],
        [ 1.4454e-01, -9.1123e-01,  1.9833e-01,  1.1301e+00, -3.2812e-01,
          7.5871e-01, -7.4364e-02,  5.5488e-01],
        [ 2.5172e-01, -4.5287e-01,  1.9549e+00, -6.9494e-01,  3.3316e-01,
         -8.5510e-01, -3.0372e-01,  1.2265e+00]], grad_fn=<ViewBa

In [ ]:
B = 2  # 每批取 2 条片段
T = 4  # 每条片段预测 4 个位置

starts = torch.randint(0, len(encoded) - T, size=(B,))
print(starts)
x_rows = []
y_rows = []

for start in starts.tolist():
    window = encoded[start : start + T + 1]  # 连续取 5 个字符 ID
    x_rows.append(torch.tensor(window[:-1], dtype=torch.long))
    y_rows.append(torch.tensor(window[1:], dtype=torch.long))

x_batch = torch.stack(x_rows)
y_batch = torch.stack(y_rows)

print("随机起点:", starts.tolist())
print("x_batch:", x_batch, "shape:", x_batch.shape)
print("y_batch:", y_batch, "shape:", y_batch.shape)

for i in range(B):
    x_text = "".join(itos[token] for token in x_batch[i].tolist())
    y_text = "".join(itos[token] for token in y_batch[i].tolist())
    print(f"第 {i + 1} 条: 输入={x_text!r}, 目标={y_text!r}")

tensor([5, 5])
随机起点: [5, 5]
x_batch: tensor([[0, 7, 5, 6],
        [0, 7, 5, 6]]) shape: torch.Size([2, 4])
y_batch: tensor([[7, 5, 6, 4],
        [7, 5, 6, 4]]) shape: torch.Size([2, 4])
第 1 条: 输入=' wor', 目标='worl'
第 2 条: 输入=' wor', 目标='worl'


In [7]:
model = nn.Embedding(8,8)
optimizer = torch.optim.SGD(model.parameters(),lr=0.1)
loss_api = nn.CrossEntropyLoss()
B , T = 2 , 4

full_x = torch.tensor(encoded[:-1], dtype=torch.long)
full_y = torch.tensor(encoded[1:], dtype=torch.long)

with torch.no_grad():
    print("Initial full-text loss:", loss_api(model(full_x), full_y).item())

for step in range(1000):
    optimizer.zero_grad(set_to_none=True)
    starts = torch.randint(0,len(encoded) - T, size=(B,))
    if step < 3:
        print(f"Step {step + 1}, starts: {starts.tolist()}")
    x1 = []
    y1 = []
    for start in starts:
        window = encoded[start:start+T+1]
        x1.append(torch.tensor(window[:-1], dtype=torch.long))
        y1.append(torch.tensor(window[1:] , dtype= torch.long))
    x_batch = torch.stack(x1)
    y_batch = torch.stack(y1)

    flat_y = y_batch.reshape(-1)
    pred = model(x_batch)
    flat_pred = pred.reshape(-1,8)
    loss = loss_api(flat_pred,flat_y)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    print("Full-text loss:", loss_api(model(full_x), full_y).item())

Initial full-text loss: 2.5530271530151367
Step 1, starts: [5, 6]
Step 2, starts: [0, 3]
Step 3, starts: [0, 2]
Full-text loss: 0.6762007474899292
